# Spotify Liked Songs — Data Wrangling + Analysis

This notebook cleans and explores my Spotify liked songs dataset. The raw export contains many columns and some messy fields (like lists stored as text). The goal is to create a clean, structured dataset that is easier to analyze.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

raw_path = "../data/raw/Liked_Songs.csv"
df = pd.read_csv(raw_path)

df.head()

In [ ]:
df.shape, df.columns.tolist()

## Cleaning decisions

- Keep the columns that are useful for analysis (track/artist/album/date + a few audio features).
- Convert dates to real datetime.
- Convert duration from milliseconds to minutes.
- Handle missing values in numeric columns.
- Remove obvious duplicates.

In [ ]:
# Keep a smaller set of columns (these exist in most Spotify exports)
keep_cols = [
    'Track Name',
    'Artist Name(s)',
    'Album Name',
    'Added At',
    'Release Date',
    'Popularity',
    'Genres',
    'Duration (ms)',
    'Danceability',
    'Energy',
    'Tempo'
]

# Only keep columns that actually exist in your file
keep_cols = [c for c in keep_cols if c in df.columns]
df2 = df[keep_cols].copy()

# Rename to snake_case
rename_map = {
    'Track Name': 'track',
    'Artist Name(s)': 'artist',
    'Album Name': 'album',
    'Added At': 'added_at',
    'Release Date': 'release_date',
    'Popularity': 'popularity',
    'Genres': 'genres',
    'Duration (ms)': 'duration_ms',
    'Danceability': 'danceability',
    'Energy': 'energy',
    'Tempo': 'tempo'
}
df2 = df2.rename(columns=rename_map)

# Dates
if 'added_at' in df2.columns:
    df2['added_at'] = pd.to_datetime(df2['added_at'], errors='coerce')

if 'release_date' in df2.columns:
    df2['release_date'] = pd.to_datetime(df2['release_date'], errors='coerce')

# Duration
if 'duration_ms' in df2.columns:
    df2['duration_min'] = df2['duration_ms'] / 60000

# Numeric cleanup
for col in ['popularity', 'danceability', 'energy', 'tempo']:
    if col in df2.columns:
        df2[col] = pd.to_numeric(df2[col], errors='coerce')

# Drop duplicates based on track + artist + album (common approach)
key_cols = [c for c in ['track', 'artist', 'album'] if c in df2.columns]
if key_cols:
    df2 = df2.drop_duplicates(subset=key_cols)

df2.head()

In [ ]:
df2.info()

## Quick exploration

In [ ]:
# Top artists
if 'artist' in df2.columns:
    df2['artist'].value_counts().head(10)

In [ ]:
# Popularity distribution
if 'popularity' in df2.columns:
    df2['popularity'].describe()

In [ ]:
# Average audio features (if present)
cols = [c for c in ['danceability', 'energy', 'tempo', 'duration_min'] if c in df2.columns]
df2[cols].mean(numeric_only=True)

## Save cleaned dataset

The cleaned dataset is saved to `data/clean/cleaned_liked_songs.csv` using a **relative path** so it works for anyone who forks this repo.

In [ ]:
out_path = "../data/clean/cleaned_liked_songs.csv"
df2.to_csv(out_path, index=False)
out_path